# 01 · Data discovery: NYC High-Volume For-Hire Vehicle trips

**Question:** can the public TLC trip records support a trusted analytics product for a
rideshare company's NYC market, and what has to be fixed or flagged first?

**Data:** every Uber (`HV0003`) and Lyft (`HV0005`) trip in New York City, October 2024 to
March 2025, from the [NYC TLC trip record data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).
The window is centred on the start of Manhattan congestion pricing (5 January 2025).

Inputs: `data/profile/` (written by `scripts/00_get_and_profile_data.py`) plus direct DuckDB
queries over the raw parquet files. Every finding here becomes a rule in
`docs/data_quality_rules.md` and a flag in `sql/silver_trips.sql`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import duckdb
import pandas as pd
import plotly.express as px

PROFILE = ROOT / "data" / "profile"
RAW = f"read_parquet('{ROOT}/data/raw/fhvhv_tripdata_*.parquet', union_by_name = true, filename = true)"
COMPANY = "case hvfhs_license_num when 'HV0003' then 'Uber' when 'HV0005' then 'Lyft' end"
con = duckdb.connect()
pd.options.display.float_format = "{:,.3f}".format

## 1. Volume and market share

In [2]:
monthly = pd.read_csv(PROFILE / "monthly_by_company.csv")
monthly["share"] = monthly["trips"] / monthly.groupby("file_month")["trips"].transform("sum")
print(f"Total trips: {monthly['trips'].sum():,}")
monthly.pivot(index="file_month", columns="company", values=["trips", "share"])

Total trips: 121,366,672


trips                share      
company             Lyft           Uber  Lyft  Uber
file_month                                         
2024-10    4,882,969.000 15,145,313.000 0.244 0.756
2024-11    4,860,617.000 15,126,916.000 0.243 0.757
2024-12    5,137,337.000 15,931,514.000 0.244 0.756
2025-01    5,049,211.000 15,356,455.000 0.247 0.753
2025-02    4,919,838.000 14,419,623.000 0.254 0.746
2025-03    5,989,698.000 14,547,181.000 0.292 0.708

In [3]:
fig = px.bar(monthly, x="file_month", y="trips", color="company", barmode="group",
             title="Monthly completed trips", labels={"file_month": "", "trips": "Trips"})
fig.add_vline(x=2.5, line_dash="dot")
fig.add_annotation(x=2.5, y=monthly["trips"].max(), text="Congestion pricing starts 5 Jan", showarrow=False, xanchor="left")
fig.show()

## 2. Schema drift between monthly files

The files are not a stable contract. A column appears mid-window, which would break a
pipeline that reads the first file's schema and assumes the rest match.

In [4]:
schema = pd.read_csv(PROFILE / "schema_by_month.csv")
presence = schema.assign(present="✓").pivot(index="column_name", columns="month", values="present").fillna("—")
presence[(presence == "—").any(axis=1)]

month,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03
column_name,,,,,,
cbd_congestion_fee,—,—,—,✓,✓,✓


**Finding:** `cbd_congestion_fee` exists only from January 2025. The pipeline reads with
`union_by_name` and treats a missing fee as 0 (DQ-08).

## 3. Fields one company doesn't report

`on_scene_datetime` (driver arrival) and `originating_base_num` are almost always empty for
Lyft, until Lyft starts sending `on_scene_datetime` partway through March 2025.

In [5]:
anom = pd.read_csv(PROFILE / "anomalies_by_month.csv")
anom["on_scene_missing_pct"] = anom["on_scene_missing"] / anom["trips"]
anom.pivot(index="file_month", columns="company", values="on_scene_missing_pct").style.format("{:.1%}")

company,Lyft,Uber
file_month,,
2024-10,99.7%,0.0%
2024-11,99.7%,0.0%
2024-12,99.7%,0.0%
2025-01,99.7%,0.0%
2025-02,99.7%,0.0%
2025-03,15.5%,0.0%


In [6]:
lyft_on_scene = con.execute(f"""
    select cast(pickup_datetime as date) as day,
           avg(case when on_scene_datetime is null then 1.0 else 0.0 end) as pct_missing
    from {RAW}
    where hvfhs_license_num = 'HV0005' and pickup_datetime >= '2025-02-20'
    group by 1 order by 1
""").df()
px.line(lyft_on_scene, x="day", y="pct_missing", title="Lyft: share of trips with no driver-arrival time",
        labels={"pct_missing": "Missing", "day": ""}).update_yaxes(tickformat=".0%").show()

**Decision (DQ-06):** no cross-company KPI uses `on_scene_datetime`. Service quality is
measured as *request to pickup*, which both companies report on every trip.

## 4. Data-quality anomalies as a share of trips

In [7]:
checks = ["nonpositive_duration", "trip_time_mismatch_2min", "trip_over_4h", "zero_or_negative_miles",
          "miles_over_150", "nonpositive_fare", "fare_over_1000", "negative_driver_pay",
          "driver_pay_exceeds_fare", "request_after_pickup", "wait_over_2h", "unknown_zone"]
rates = anom.groupby("company")[checks + ["trips"]].sum()
rates = (rates[checks].div(rates["trips"], axis=0)).T
rates.style.format("{:.3%}")

company,Lyft,Uber
nonpositive_duration,0.009%,0.008%
trip_time_mismatch_2min,0.018%,0.015%
trip_over_4h,0.001%,0.001%
zero_or_negative_miles,0.004%,0.017%
miles_over_150,0.000%,0.001%
nonpositive_fare,0.351%,0.031%
fare_over_1000,0.000%,0.000%
negative_driver_pay,0.000%,0.002%
driver_pay_exceeds_fare,1.107%,3.454%
request_after_pickup,1.680%,0.832%


## 5. Daylight saving time breaks timestamp arithmetic

TLC timestamps are local New York time with no time-zone offset. Trips that span a clock
change get a timestamp duration that is an hour off.

In [8]:
dst = con.execute(f"""
    select cast(pickup_datetime as date) as day, {COMPANY} as company,
           count(*) as trips_with_mismatch
    from {RAW}
    where abs(trip_time - date_diff('second', pickup_datetime, dropoff_datetime)) > 120
    group by 1, 2 order by 3 desc limit 6
""").df()
dst

,day,company,trips_with_mismatch
0,2024-11-03,Uber,6931
1,2025-03-09,Uber,5860
2,2024-11-03,Lyft,2898
3,2025-03-09,Lyft,2636
4,2025-03-06,Uber,20
5,2025-03-13,Uber,16


**Finding:** almost every mismatch falls on **3 Nov 2024** (clocks back) or **9 Mar 2025**
(clocks forward). **Decision (DQ-04):** durations always come from TLC's `trip_time`
field, never from `dropoff - pickup`.

## 6. Lyft zero-fare trips spike in some months

In [9]:
zero = con.execute(f"""
    select cast(pickup_datetime as date) as day, {COMPANY} as company,
           count(*) filter (where base_passenger_fare <= 0) as zero_fare_trips,
           count(*) as trips
    from {RAW}
    group by 1, 2 order by 1
""").df()
zero["zero_fare_share"] = zero["zero_fare_trips"] / zero["trips"]
px.line(zero, x="day", y="zero_fare_share", color="company", title="Share of trips with base fare <= $0",
        labels={"zero_fare_share": "", "day": ""}).update_yaxes(tickformat=".1%").show()

**Decision (DQ-03):** zero-fare trips (probably promotions or comped rides) count as trips
but are excluded from price and margin KPIs, so they don't drag average fare down.

## 7. Driver pay above fare

In [10]:
pay = con.execute(f"""
    select {COMPANY} as company, strftime(pickup_datetime, '%Y-%m') as month,
           avg(case when driver_pay > base_passenger_fare + tips + 5 then 1.0 else 0.0 end) as share_pay_above_fare,
           sum(base_passenger_fare - driver_pay) / sum(base_passenger_fare)              as take_rate_raw
    from {RAW}
    where base_passenger_fare > 0
    group by 1, 2 order by 2, 1
""").df()
pay.pivot(index="month", columns="company", values=["share_pay_above_fare", "take_rate_raw"]).style.format("{:.1%}")

**Decision (DQ-05):** these trips are kept. They are real money (likely driver incentives),
and dropping them would overstate take rate. They are flagged `is_pay_above_fare` for analysis.

## 8. Wait time distribution

In [11]:
waits = con.execute(f"""
    select {COMPANY} as company,
           quantile_cont(date_diff('second', request_datetime, pickup_datetime) / 60.0, [0.1, 0.25, 0.5, 0.75, 0.9, 0.99]) as q
    from {RAW}
    where request_datetime <= pickup_datetime
    group by 1
""").df()
pd.DataFrame(waits["q"].tolist(), index=waits["company"], columns=["p10", "p25", "p50", "p75", "p90", "p99"])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,p10,p25,p50,p75,p90,p99
company,,,,,,
Lyft,2.167,3.100,4.467,6.333,8.833,16.083
Uber,1.917,2.717,3.917,5.717,8.333,16.250


## Summary: findings → product decisions

| # | Finding | Decision | Rule |
|---|---|---|---|
| 1 | ~4.5% of trips have an unknown or outside-NYC zone | Keep in totals, exclude from zone views | DQ-01 |
| 2 | ~1% of trips (Lyft ~1.7%) have request after pickup | Exclude from wait KPIs only | DQ-02 |
| 3 | Zero-fare trips spike for Lyft in Dec, Jan, Mar | Exclude from price and margin KPIs | DQ-03 |
| 4 | DST makes timestamp durations wrong | Use `trip_time`, never timestamp maths | DQ-04 |
| 5 | Driver pay > fare on 2–5% of Uber trips, ~1% of Lyft | Keep and flag | DQ-05 |
| 6 | Lyft doesn't report driver arrival until Mar 2025 | Use request-to-pickup for service KPIs | DQ-06 |
| 7 | Files can hold trips from adjacent months | Attribute by pickup date | DQ-07 |
| 8 | `cbd_congestion_fee` added Jan 2025 | Read by name, missing = 0 | DQ-08 |